# <center> **Car Brands** 🚙 🚗 🚕
    
Car brand classification model trained with the following datasets:
    
- train-data/source-1/: [+300 car brands dataset](https://www.kaggle.com/datasets/alirezaatashnejad/over-20-car-brands-dataset/code)
- train-data/source-2/: [UK car brands dataset](https://www.kaggle.com/datasets/bignosethethird/uk-car-brands-dataset?select=UKCarsDataset.1005)
- train-data/source-3/: [Car brands images](https://www.kaggle.com/datasets/yamaerenay/100-images-of-top-50-car-brands)
- train-data/source-4/: [Car brands in Egypt](https://www.kaggle.com/datasets/mohamedaziz15/cars-brands-in-egypt?select=Hundai)
---

In [1]:
!pip install tensorflow seaborn mlflow openvino -q


[notice] A new release of pip available: 22.2.2 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# model configuration for this run...
DIR = "train-data/"

IMAGE_SIZE = (300, 300)
INPUT_SHAPE = IMAGE_SIZE+(3,)

_2ND_STEP = True
VAL_SPLIT = 0.2
SEED = 42

model_name = "brands"
model_version = "3"

## Config *mlflow* Run

## Define classes

In [3]:
import os
import shutil

# remove .ipynb_checkpoints from main dir
try: shutil.rmtree(DIR+".ipynb_checkpoints")
except FileNotFoundError: pass

# remove .ipynb_checkpoint from subdirs
for brand in os.listdir(DIR):
    path = os.path.join(DIR, brand)
    try: shutil.rmtree(os.path.join(path, ".ipynb_checkpoints"))
    except FileNotFoundError: continue

In [4]:
import glob

# get classes
brands = os.listdir(DIR)

# count number of classes
n = len(brands)

# count number of images
files = glob.glob(os.path.join(DIR, '**', '*'), recursive=True)
files = [f for f in files if os.path.isfile(f)]
num_files = len(files)

# print final DIR structure
print(DIR)
for brand in brands:
    count = len(os.listdir(os.path.join(DIR, brand)))
    print(f"\t{brand}: {count}")
print(f"\nTotal of {n} classes")
print(f"Total of {num_files} images")

train-data/
	Kia: 1000
	Mercedes-Benz: 1000
	Bmw: 1000
	Hyundai: 1000
	Toyota: 1000

Total of 5 classes
Total of 5000 images


## Image Data Generators
We will use `ImageDataGenerator` class from TensorFlow's Keras API, for...

**Preprocessing**: 
- resize images to (300, 300) while maintaining aspect ratio
- sets aside 20% of the training data for validation
        
**Data augmentation** (only for train data):
- randomly rotate images within -10 to 10 degrees
- randomly shift images horizontally by up to 10% of the width
- randomly shift images vertically by up to 10% of the height
- randomly applies shearing transformations up to 10%
- randomly zoom in or out on images by up to 10%
- randomly flip images horizontally, not vertically
- uses the nearest pixel value to fill in new pixels
        
We will set batch size to 128 for training data and 64 for validation data.

>***WARNING ⚠️:*** 
>
> *Read the following [issue#5862](https://github.com/keras-team/keras/issues/5862#issuecomment-647559571) about how to split train and valid data while **keeping augmentation changes only for training data**.*

In [5]:
import numpy as np
import tensorflow as tf

# function to resize images while maintaining aspect ratio
def resize_with_aspect_ratio(image, target_size):
    target_height, target_width = target_size
    image_shape = tf.cast(tf.shape(image)[:2], tf.float32)  # Ensure the shape is float32 for calculation
    height, width = image_shape[0], image_shape[1]

    scale = tf.minimum(target_width / width, target_height / height)

    new_height = tf.cast(height * scale, tf.int32)  # Cast back to int32 after calculation
    new_width = tf.cast(width * scale, tf.int32)    # Cast back to int32 after calculation

    resized_image = tf.image.resize(image, (new_height, new_width))

    # Pad the image to target size
    padded_image = tf.image.pad_to_bounding_box(
        resized_image,
        offset_height=(target_height - new_height) // 2,
        offset_width=(target_width - new_width) // 2,
        target_height=target_height,
        target_width=target_width
    )

    return padded_image

# custom preprocessing function
def preprocess_image(image):
    image = resize_with_aspect_ratio(image, IMAGE_SIZE)
    return image


2024-08-11 21:47:03.873691: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-08-11 21:47:03.874329: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-11 21:47:03.877675: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-11 21:47:03.888870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-11 21:47:03.907559: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# create image generators for training data
datagen_train = ImageDataGenerator(
    # preprocessing:
    preprocessing_function=preprocess_image,
    validation_split=VAL_SPLIT,
    # data augmentation:
    horizontal_flip=True,
    vertical_flip=False,
    fill_mode='nearest',
    height_shift_range=0.2,
    width_shift_range=0.2,
    rotation_range=20,
    shear_range=0.2,
    zoom_range=0.2
)

train_generator = datagen_train.flow_from_directory(
    target_size=IMAGE_SIZE,
    directory=DIR,
    seed=SEED,
    class_mode='categorical',
    subset='training',
    batch_size=64,
    shuffle=True
)

# create image generators for validation data
datagen_val = ImageDataGenerator(
    preprocessing_function=preprocess_image,
    validation_split=VAL_SPLIT
)

val_generator = datagen_val.flow_from_directory(
    target_size=IMAGE_SIZE,
    directory=DIR,
    seed=SEED,
    class_mode='categorical',
    subset='validation',
    batch_size=32,
    shuffle=False
)

Found 4000 images belonging to 5 classes.
Found 1000 images belonging to 5 classes.


In [7]:
# store img names in valid_set.txt and train_set.txt
train_file = 'train_set.txt'
val_file = 'valid_set.txt'

def save_file_names(generator, save_path):
    with open(save_path, 'w') as f:
        for filepath in generator.filepaths:
            f.write(f"{os.path.basename(filepath)}\n")

save_file_names(train_generator, train_file)
save_file_names(val_generator, val_file)

## Transfer Learning

We will leverage the feature extraction capabilities of a pre-trained model (***InceptionV3***), which has already learned good feature extraction capabilities. In order to do that we will need to ...

1. ...**drop last few layers** (optional)

2. ...**freeze bottom layers**, because we do not want to re-train them

3. ...**add dense layers at the top** of the network for car-brand classification
    
Our CNN architecture will loke something like this:

    - input layer: (300, 300)
    - scale layer: normalize the pixel values to the [0, 1] range
    - base model: InceptionV3 (without top layers)
    - flatten layer
    - dense layer (softmax): 10 units, with 50% dropout

>***NOTE:*** 
>
> *[Keras 3 API documentation / Keras Applications](https://keras.io/api/applications/#usage-examples-for-image-classification-models)*

In [8]:
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras import layers, Model, Input
from tensorflow.keras import Model

# load pre-trained model
base_model = InceptionV3(
    input_shape=INPUT_SHAPE,
    include_top=False,
    weights="imagenet"
)

# drop last few layers (optional)
layer_name_to_keep = "mixed8"
model_output = base_model.get_layer(layer_name_to_keep).output
base_model = Model(inputs=base_model.input, outputs=model_output)

# freeze the base_model
base_model.trainable = False

# pre-trained Inception weights requires that input be scaled
# from (0, 255) to a range of (-1., +1.), the rescaling layer
# outputs: `(inputs * scale) + offset`
inputs = Input(shape=INPUT_SHAPE)
scale_layer = layers.Rescaling(scale=1/127.5, offset=-1)

# we make sure to pass training=False when calling the base model,
# so that it runs in inference mode, so that batchnorm statistics don't get
# updated even after we unfreeze the base model for fine-tuning.
x = scale_layer(inputs)
x = base_model(x, training=False)

# add layers for car.brand classification on top...
x = layers.Flatten()(x) # Flatten or GlobalAveragePooling2D ?
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(units=100, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.25)(x)
x = layers.Dense(units=n, activation="softmax")(x)

model = Model(inputs, x)

# print model summary
model.summary(show_trainable=True)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1 (InputLayer)  │ (None, 300, 300, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ rescaling (Rescaling)       │ (None, 300, 300, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ functional (Functional)     │ (None, 8, 8, 1280)    │ 10,674,848 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ flatten (Flatten)           │ (None, 81920)         │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ batch_normalization_94      │ (None, 81920)         │    327,680 │   Y   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout (Dropout)           │ (None, 81920)         │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense (Dense)               │ (None, 100)           │  8,192,100 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ batch_normalization_95      │ (None, 100)           │        400 │   Y   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout_1 (Dropout)         │ (None, 100)           │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense_1 (Dense)             │ (None, 5)             │        505 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 19,195,533 (73.23 MB)

 Trainable params: 8,356,645 (31.88 MB)

 Non-trainable params: 10,838,888 (41.35 MB)

## Callbacks

We are going to define some callbacks to monitor validation scores during training and intervene if necessary:

- `TargetAccuracy`: if accuracy is greater than 95%, then stop training
- `ReduceLROnPlateau`: if loss does not improve for 2 epochs,
                     then $lr´=\max(0.5\times lr ; 0.00001)$
- `ModelCheckpoint`: save weights for the best model so far
- `EarlyStopping`: if loss doesn´t improve for 5 epochs, then stop training

In [9]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    min_lr=0.00001,
    factor=0.25,
    patience=2,
    verbose=True
)

early_stopping = EarlyStopping(
    restore_best_weights=True,
    monitor="val_accuracy",
    patience=5,
)

callbacks = [reduce_lr, early_stopping]

## Model Training
We will train the model for **30 epochs**, and using **categorial-crossentropy** loss function with ***ADAM*** optimizer algorithm.

In [10]:
%%time
import keras

# compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# train the top layers only
history_ = model.fit(
    train_generator,
    validation_data=val_generator,
    callbacks=callbacks,
    epochs=15
)

Epoch 1/15


/opt/app-root/lib64/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
2024-08-11 21:47:26.355235: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:2: Filling up shuffle buffer (this may take a while): 7 of 8
2024-08-11 21:47:27.831012: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


63/63 ━━━━━━━━━━━━━━━━━━━━ 392s 6s/step - accuracy: 0.3844 - loss: 1.9431 - val_accuracy: 0.3360 - val_loss: 1.7005 - learning_rate: 0.0010
Epoch 2/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 393s 6s/step - accuracy: 0.5469 - loss: 1.1995 - val_accuracy: 0.3330 - val_loss: 1.5724 - learning_rate: 0.0010
Epoch 3/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 387s 6s/step - accuracy: 0.6178 - loss: 1.0143 - val_accuracy: 0.4170 - val_loss: 1.4402 - learning_rate: 0.0010
Epoch 4/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 387s 6s/step - accuracy: 0.6460 - loss: 0.9301 - val_accuracy: 0.4400 - val_loss: 1.4559 - learning_rate: 0.0010
Epoch 5/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 390s 6s/step - accuracy: 0.6719 - loss: 0.8691 - val_accuracy: 0.4450 - val_loss: 1.4272 - learning_rate: 0.0010
Epoch 6/15


2024-08-11 22:19:50.363648: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:2: Filling up shuffle buffer (this may take a while): 7 of 8
2024-08-11 22:19:51.832192: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


63/63 ━━━━━━━━━━━━━━━━━━━━ 386s 6s/step - accuracy: 0.6932 - loss: 0.8233 - val_accuracy: 0.4650 - val_loss: 1.3596 - learning_rate: 0.0010
Epoch 7/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 373s 6s/step - accuracy: 0.6949 - loss: 0.7901 - val_accuracy: 0.4780 - val_loss: 1.4574 - learning_rate: 0.0010
Epoch 8/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.7111 - loss: 0.7456
Epoch 8: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
63/63 ━━━━━━━━━━━━━━━━━━━━ 375s 6s/step - accuracy: 0.7108 - loss: 0.7460 - val_accuracy: 0.4850 - val_loss: 1.4563 - learning_rate: 0.0010
Epoch 9/15


2024-08-11 22:38:44.255135: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:2: Filling up shuffle buffer (this may take a while): 7 of 8
2024-08-11 22:38:45.621166: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


63/63 ━━━━━━━━━━━━━━━━━━━━ 376s 6s/step - accuracy: 0.7149 - loss: 0.7332 - val_accuracy: 0.4910 - val_loss: 1.3923 - learning_rate: 2.5000e-04
Epoch 10/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.7278 - loss: 0.6963
Epoch 10: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
63/63 ━━━━━━━━━━━━━━━━━━━━ 396s 6s/step - accuracy: 0.7279 - loss: 0.6962 - val_accuracy: 0.4900 - val_loss: 1.3991 - learning_rate: 2.5000e-04
Epoch 11/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 381s 6s/step - accuracy: 0.7542 - loss: 0.6455 - val_accuracy: 0.4940 - val_loss: 1.3959 - learning_rate: 6.2500e-05
Epoch 12/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.7359 - loss: 0.6807
Epoch 12: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.
63/63 ━━━━━━━━━━━━━━━━━━━━ 370s 6s/step - accuracy: 0.7360 - loss: 0.6804 - val_accuracy: 0.4940 - val_loss: 1.3912 - learning_rate: 6.2500e-05
Epoch 13/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 385s 6s/step - accuracy: 0.7461 - loss: 0.6649 - va

## Fine-Tuning
A last, optional step, is fine-tuning, which consists of **unfreezing** the entire model you obtained above (or part of it), and **re-training** it on the new data with a very low learning rate. 

This can potentially achieve meaningful improvements, by incrementally adapting the pretrained features to the new data.

>***NOTE:*** 
>
> *[Developer guides / Transfer learning & fine-tuning](https://keras.io/guides/transfer_learning/)*

In [12]:
%%time

if _2ND_STEP:
    # unfreeze some top layers of the base_model 
    for layer in base_model.layers[:165]:
        layer.trainable = False
    for layer in base_model.layers[165:]:
        layer.trainable = True

    # another option is to unfreeze the entire base_model: base_model.trainable = True

    model.summary(show_trainable=True)
    # note that it keeps running in inference mode since we passed `training=False` when calling it.
    # this means that the batchnorm layers will not update their batch statistics.
    # this prevents the batchnorm layers from undoing all the training we've done so far.

    # we need to compile the model again
    model.compile(
        optimizer=keras.optimizers.Adam(1e-5), # low learning rate
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    # start mlflow run
    #mlflow.keras.autolog() 

    # fine-tune of trainable layers
    history_ = model.fit(train_generator,
                         validation_data=val_generator,
                         epochs=10
                        )
    # end mlflow run
    #mlflow.end_run()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1 (InputLayer)  │ (None, 300, 300, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ rescaling (Rescaling)       │ (None, 300, 300, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ functional (Functional)     │ (None, 8, 8, 1280)    │ 10,674,848 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ flatten (Flatten)           │ (None, 81920)         │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ batch_normalization_94      │ (None, 81920)         │    327,680 │   Y   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout (Dropout)           │ (None, 81920)         │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense (Dense)               │ (None, 100)           │  8,192,100 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ batch_normalization_95      │ (None, 100)           │        400 │   Y   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout_1 (Dropout)         │ (None, 100)           │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense_1 (Dense)             │ (None, 5)             │        505 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 35,908,825 (136.98 MB)

 Trainable params: 13,882,981 (52.96 MB)

 Non-trainable params: 5,312,552 (20.27 MB)

...

2024-08-12 02:21:43.302484: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:41: Filling up shuffle buffer (this may take a while): 7 of 8
2024-08-12 02:21:44.856855: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


63/63 ━━━━━━━━━━━━━━━━━━━━ 558s 9s/step - accuracy: 0.7174 - loss: 0.7348 - val_accuracy: 0.5070 - val_loss: 1.4085 - learning_rate: 1.0000e-05
Epoch 2/10


2024-08-12 02:30:50.196520: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:41: Filling up shuffle buffer (this may take a while): 7 of 8
2024-08-12 02:30:51.604876: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.7381 - loss: 0.6793
Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
63/63 ━━━━━━━━━━━━━━━━━━━━ 545s 8s/step - accuracy: 0.7381 - loss: 0.6793 - val_accuracy: 0.5110 - val_loss: 1.4316 - learning_rate: 1.0000e-05
Epoch 3/10
44/63 ━━━━━━━━━━━━━━━━━━━━ 2:22 7s/step - accuracy: 0.7405 - loss: 0.6676


KeyboardInterrupt



## Model Evaluation
First, we will plot the **learning curves**...

In [ ]:
# set up plotting tools
import matplotlib.pyplot as plt
plt.style.use('bmh')

In [ ]:
# plot loss learning curves
loss = history_.history['loss']
val_loss = history_.history['val_loss']
epochs = range(len(loss))

ax, fig = plt.subplots(figsize=(7,4))
plt.plot(epochs, loss, color='crimson', label='Training')
plt.plot(epochs, val_loss, color='orange', label='Validation')
plt.title('Loss Learning Curves')
plt.legend();

In [ ]:
# plot accuracy learning curves
acc = history_.history['accuracy']
val_acc = history_.history['val_accuracy']
epochs = range(len(acc))

ax, fig = plt.subplots(figsize=(7,4))
plt.plot(epochs, acc, color='green', label='Training')
plt.plot(epochs, val_acc, color='deepskyblue', label='Validation')
plt.title('Accuracy Learning Curves')
plt.legend();

Second, we will create the **classification report**...

In [ ]:
import numpy as np

# make predictions on validation data
predictions = model.predict(val_generator)
predictions = np.argmax(predictions, axis=-1)

# get model categories
model_classes = list(val_generator.class_indices.keys())

In [ ]:
# create classification report
from sklearn.metrics import classification_report

report = classification_report(val_generator.labels,
                               predictions,
                               target_names=model_classes
                              )
display(report)

Finally, we will plot the **confusion matrix**...

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

# create confusion matrix
cm = confusion_matrix(val_generator.labels, predictions)
cm_df = pd.DataFrame(cm,
                     index=model_classes,
                     columns=model_classes
                    )

# plot confusion matrix
plt.figure(figsize=(8,5))
plt.title('Confusion Matrix')
sns.heatmap(cm_df,
            fmt='',
            annot=True,
            linewidth=1,
            cmap="Greens",
            linecolor='black'
           );

In [ ]:
# plot confusion matrix in %
cm_normalized = cm.astype('float')/cm.sum(axis=1)[:, np.newaxis]
cm_normalized_df = pd.DataFrame(cm_normalized,
                                index=model_classes,
                                columns=model_classes
                               )

plt.figure(figsize=(8,5))
plt.title('Confusion Matrix')
sns.heatmap(cm_normalized_df,
            fmt='.0%',
            annot=True,
            linewidth=1,
            cmap="Greens",
            linecolor='black'
           );

## Predictions Sample

In [ ]:
label2id = val_generator.class_indices
id2label = {v:k for k,v in label2id.items()}

# pick a random image from the validation set,
# print its true label and predicted label.
import random
img_id = random.randint(0, len(predictions)-1)

pred_class = id2label[predictions[img_id]]
true_class = id2label[val_generator.labels[img_id]]
print(f"Predicted: {pred_class}\nActual: {true_class}")

img_path = os.path.join(DIR, val_generator.filenames[img_id])
img = plt.imread(img_path)
plt.imshow(img);

In [ ]:
# display images that were misclassified in the validation set
misclassified_indices = np.where(predicted_classes != true_classes)[0]

def show_misclassified_images(generator, misclassified_indices, predicted_classes, true_classes):
    for index in misclassified_indices:
        image_path = generator.filepaths[index]
        img = plt.imread(image_path)

        true_class = generator.class_indices[true_classes[index]]
        pred_class = generator.class_indices[predicted_classes[index]]
        title = f"Predicted: {pred_class}\nActual: {true_class}"

        plt.figure()
        plt.imshow(img)
        plt.title(title)
        plt.axis('off')
        plt.show()

show_misclassified_images(val_generator,
                          misclassified_indices,
                          predicted_classes=predictions,
                          true_classes=val_generator.labels)

## Convert *TensorFlow* model to *OpenVINO* format

---